# Real PIA ICAO mappings

Add the real ICAO from OpenAirframes by joining the Jacksweeney tail-number mapping on `Tail Number`.

In [1]:
from pathlib import Path
import sys
import polars as pl
from data_engineering.openairframes.read import scan_latest_faa_registration_icao_for_join

In [32]:
mappings_path = Path("/Users/jonahgoode/Documents/PlaneQuery/r-piaicao jacksweeney mappings.csv")

df_mappings = pl.read_csv(mappings_path).rename(
    {
        "PIA ICAO": "pia_icao",
        "Tail Number": "tailnumber",
        "Date": "date",
    }
).with_columns(
    pl.col("pia_icao").str.strip_chars().str.to_lowercase(),
    pl.col("tailnumber").str.strip_chars().str.to_uppercase(),
    pl.col("date").str.strptime(pl.Date, strict=False),
)
df_real_icaos = scan_latest_faa_registration_icao_for_join().collect()

df_mappings = df_mappings.join(
    df_real_icaos,
    left_on="tailnumber",
    right_on="registration",
    how="left",
)
pia_to_real_icao_map = dict(zip(df_mappings["pia_icao"], df_mappings["real_icao"]))

In [39]:
from datetime import date, timedelta
from data_engineering.flights.flight_type import get_flights

start_date = date(2026, 6, 1)
end_date = date(2026, 6, 8)

df_flights = get_flights(start_date, end_date)

In [40]:
pia_icaos = df_mappings.select(pl.col("pia_icao")).unique().to_series().to_list()
df_matched_flights = df_flights.filter(pl.col("icao").is_in(pia_icaos))

In [41]:
from data_engineering.flights.flight_type import flights_df_to_flights
flights = flights_df_to_flights(df_matched_flights)

In [42]:
pia_mapping_dict = dict()
for flight in flights:
    pia_mapping_dict[flight.flight_id] = pia_to_real_icao_map[flight.icao]

In [44]:
df_flights.filter(pl.col("pia"))

icao,callsign,registration,takeoff_time,takeoff_airport_ident,takeoff_airport_ident_distance_km,landing_time,landing_airport_ident,landing_airport_ident_distance_km,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,takeoff_airport_elevation_difference_ft,landing_airport_elevation_difference_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,f64,datetime[ms],str,f64,f64,f64,i64,i64,i64,i64,bool,bool,bool,bool,str,str,str,str
"""a03954""","""XAA8796""",null,2026-06-01 20:18:09.610,"""KFLL""",null,2026-06-01 20:26:03.820,"""KMIA""",null,null,null,null,null,null,null,true,false,false,false,null,null,null,"""A3"""
"""a0398c""","""XAA8583""",null,2026-06-01 18:29:09.100,"""KBDL""",null,2026-06-01 21:01:18.510,"""KMEM""",null,null,null,null,null,null,null,true,false,false,false,null,null,null,"""A2"""
"""a0398c""","""XAA4772""",null,2026-06-01 22:01:34.960,"""KMEM""",null,2026-06-02 00:11:50.270,"""4B9""",null,null,null,null,null,null,null,true,false,false,false,null,null,null,"""A2"""
"""a0399f""","""FFL1320""",null,2026-06-01 14:59:01.750,"""KFLD""",null,2026-06-01 18:32:50.730,"""KMRY""",null,null,null,null,null,null,null,true,false,false,false,null,null,null,"""A3"""
"""a03b9f""","""N48FW""",null,2026-06-01 17:22:50.910,"""KFKR""",null,2026-06-01 17:34:01.860,"""KTYQ""",null,null,null,null,null,null,null,true,false,false,false,null,null,null,"""A1"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""a10158""","""FFL1064""",null,2026-06-07 14:09:25.080,"""KHEF""",null,2026-06-07 14:40:51.910,"""MD45""",null,null,null,null,null,null,null,true,false,false,false,null,null,null,"""A2"""
"""a1c15a""","""N212EF""",null,2026-06-07 14:14:54.720,"""OH38""",null,2026-06-07 14:26:08.910,"""OH38""",null,null,null,null,null,null,null,true,false,false,false,null,null,null,"""A7"""
"""a4e954""","""FFL645""",null,2026-06-07 01:14:02.780,"""KIWA""",null,2026-06-07 01:49:07.850,"""KE91""",null,null,null,null,null,null,null,true,false,false,false,null,null,null,"""A1"""


In [43]:
df_flights.filter(pl.col("icao").is_in(pia_to_real_icao_map.keys()))

icao,callsign,registration,takeoff_time,takeoff_airport_ident,takeoff_airport_ident_distance_km,landing_time,landing_airport_ident,landing_airport_ident_distance_km,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,takeoff_airport_elevation_difference_ft,landing_airport_elevation_difference_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,f64,datetime[ms],str,f64,f64,f64,i64,i64,i64,i64,bool,bool,bool,bool,str,str,str,str
"""a0398c""","""XAA8583""",null,2026-06-01 18:29:09.100,"""KBDL""",null,2026-06-01 21:01:18.510,"""KMEM""",null,null,null,null,null,null,null,true,false,false,false,null,null,null,"""A2"""
"""a0398c""","""XAA4772""",null,2026-06-01 22:01:34.960,"""KMEM""",null,2026-06-02 00:11:50.270,"""4B9""",null,null,null,null,null,null,null,true,false,false,false,null,null,null,"""A2"""
"""a03b9f""","""N48FW""",null,2026-06-01 17:22:50.910,"""KFKR""",null,2026-06-01 17:34:01.860,"""KTYQ""",null,null,null,null,null,null,null,true,false,false,false,null,null,null,"""A1"""
"""a03c1f""","""XAA571""",null,2026-06-01 00:59:41.990,"""KSJC""",null,2026-06-01 01:49:45.020,"""KLAX""",null,null,null,null,null,null,null,true,false,false,false,null,null,null,"""A3"""
"""a03c1f""","""XAA4320""",null,2026-06-01 02:39:25.890,"""KLAX""",null,2026-06-01 03:23:06.190,"""KLAS""",null,null,null,null,null,null,null,true,false,false,false,null,null,null,"""A3"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""a100dc""","""FFL885""",null,2026-06-07 16:07:58.300,"""KPWK""",null,2026-06-07 17:07:14.420,"""KSDF""",null,null,null,null,null,null,null,true,false,false,false,null,null,null,"""A2"""
"""a10100""","""FFL1507""",null,2026-06-07 17:26:35.220,"""KHIO""",null,2026-06-07 20:01:53.920,"""KLAM""",null,null,null,null,null,null,null,true,false,false,false,null,null,null,"""A1"""
"""a10100""","""FFL1507""",null,2026-06-07 22:06:00.530,"""KSAF""",null,2026-06-08 00:25:04.960,"""KRST""",null,null,null,null,null,null,null,true,false,false,false,null,null,null,"""A1"""
